# PEFT Training — LoRA on Qwen2.5-0.5B (Week 3)

Dedicated notebook for parameter-efficient fine-tuning experiments (LoRA now, QLoRA-ready), kept separate from `sft_training.ipynb` (full fine-tuning, Week 1/2) so both stay independently re-runnable — Week 3's practical work requires comparing full FT vs. LoRA vs. QLoRA directly, and running them from the same notebook would make that harder to keep straight across sessions.

**Before running:** Settings → Accelerator → GPU T4 x2 (same as before — do not pick P100).

## 0. Restrict to a single GPU

Same reason as `sft_training.ipynb`: Kaggle's "T4 x2" auto-wraps the model in `DataParallel` when two GPUs are visible, which crashes against our single-device placement.

In [ ]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "0"

## 1. Pull the repo and install dependencies

Same private-repo sparse-checkout pattern as before — see `sft_training.ipynb` for the full explanation of why (Obsidian filenames breaking Kaggle's dataset export, only `src/` being needed here).

In [ ]:
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
github_token = secrets.get_secret("GITHUB_TOKEN")
repo_url = f"https://{github_token}@github.com/zoom-BT/llm-alignment-internship.git"

!git clone --filter=blob:none --no-checkout {repo_url}
%cd llm-alignment-internship
!git sparse-checkout init --cone
!git sparse-checkout set src
!git checkout main
!pip install -q -r requirements.txt

## 2. Confirm the GPU is visible

In [ ]:
import torch

print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))

## 3. Dry run (120 steps) — LoRA

`config.yaml` already had a `training.full_finetune` flag and a `training.lora` block (`r=16, alpha=32, dropout=0.05`) sitting there unused since Week 1. We switch it **in memory, after loading**, rather than editing the committed file — Week 1/2's full fine-tuning runs must stay reproducible exactly from `config.yaml` as checked in, so the LoRA/full-FT choice is made per-run here, not by changing the shared default.

120 steps again (not 20), for the same reason as Week 2: `eval_steps=100` needs at least one evaluation to actually fire before we trust the wiring.

In [ ]:
import yaml
from src.train import run_sft

config = yaml.safe_load(open("config.yaml"))
config["training"]["full_finetune"] = False  # switch to LoRA for this notebook's runs

trainer = run_sft(config, max_steps=120)
print("Dry run finished without OOM, and eval/early-stopping ran at least once.")
print("Trainable params:")
trainer.model.print_trainable_parameters()

## 4. Full LoRA training run

Only run once the dry run above completed cleanly. `config` still has `full_finetune=False` from the cell above (same Python session), so this reuses it — no need to redefine anything.

In [ ]:
trainer = run_sft(config)
print("LoRA training complete. Adapter saved to results/checkpoints/final")

## 4a. Display the training curves

In [ ]:
from IPython.display import Image, display

display(Image(filename="results/training_curve.png"))

## 4b. Merge the LoRA adapter, then evaluate

`trainer.save_model()` on a LoRA run saves only the **adapter** (`B`, `A` from the math we worked through — a few MB, not the full model): `AutoModelForCausalLM.from_pretrained()` in `run_benchmark()` can't load that directly, it expects full model weights. So we merge the adapter into the base model first (`W + (alpha/r)·B·A`, computed once) and save that as an ordinary checkpoint — `run_benchmark()` then needs no changes at all.

In [ ]:
merged_path = "results/checkpoints/final_merged"

merged_model = trainer.model.merge_and_unload()
merged_model.save_pretrained(merged_path)
trainer.processing_class.save_pretrained(merged_path)
print(f"Merged model saved to {merged_path}")

from src.evaluate import run_benchmark

lora_results = run_benchmark(config, model_path=merged_path, output_filename="lora_week3_results.json")
lora_results

## 4c. Zip the merged model and clean up

Two things worth keeping this time (the small adapter *and* the merged model), unlike Week 1/2's single-checkpoint case — `cleanup_checkpoint_dir()` only supports keeping one named entry, so intermediate `checkpoint-*` snapshots are removed directly here instead of reusing it.

In [ ]:
import shutil
from pathlib import Path

shutil.make_archive(merged_path, "zip", merged_path)

for path in Path("results/checkpoints").glob("checkpoint-*"):
    shutil.rmtree(path)

print("Kept results/checkpoints/final (adapter) and final_merged.zip; removed intermediate checkpoints.")

## 5. Next step

Result lands in `results/lora_week3_results.json`, directly comparable to the existing baseline (13.92), Week 1 full-FT (75.41), and Week 2 full-FT (14.61) — same dataset, split, seed, and `evaluate.py` code path throughout.

For the QLoRA run (still needed for Week 3's full-FT vs. LoRA vs. QLoRA comparison), this same notebook can be reused: load the base model with a 4-bit `BitsAndBytesConfig` (`quantization_config` — already confirmed available on `SFTTrainer`) before calling `run_sft`, keeping `full_finetune=False` and the same `lora` block unchanged.